In [8]:
from fastapi import FastAPI, status

import importlib
import src.agent.transport_agent
import src.tools.tools_definition
import src.services.session_service
import src.utils.logging
import src.config.settings
import src.mcp.client
import src.mcp.server
import src.tools.handlers
importlib.reload(src.tools.handlers)
importlib.reload(src.mcp.server)
importlib.reload(src.mcp.client)
importlib.reload(src.config.settings)
importlib.reload(src.utils.logging)
importlib.reload(src.services.session_service)
importlib.reload(src.tools.tools_definition)
importlib.reload(src.agent.transport_agent)

from src.agent.transport_agent import TransportAgent
from src.config.settings import Settings
from src.schemas import ChatResponse, MessageRequest
from src.services.session_service import SessionService
from src.utils.logging import setup_logging
from src.mcp.client import MCPClient

settings = Settings()
session_logger = setup_logging(
    level=settings.log_level,
    logger_name='SESSION',
    log_file=settings.log_file,
)
app_logger = setup_logging(
    level=settings.log_level,
    logger_name='SERVER',
    log_file=settings.log_file,
)
MCP_client_logger = setup_logging(
    level=settings.log_level,
    logger_name='MCP_CLIENT',
    log_file=settings.log_file,
)
agent_logger = setup_logging(
    level=settings.log_level,
    logger_name='AGENT',
    log_file=settings.log_file,
)
session_service = SessionService(settings.sessions_dir)
mcp_client = MCPClient()
server_script_path = 'src.mcp.server'
agent = TransportAgent(settings=settings, session_service=session_service, mcp_client=mcp_client)

In [10]:
response = await agent.run_conversation(
    'hello, check the status of package id PKG12345678',
    '1234xdd'
)
# response = await agent.run_conversation(
#     'Inside are the reactor components could you send it to the PWR3847PL?',
#     '1234xdd'
# )
# response = await agent.run_conversation(
#     'The code is 194123',
#     '1234xdd'
# )
response

'PKG12345678 — status: unknown. Still no tracking data available. Want me to try another redirect or open an investigation?'

In [18]:
response

'Hey — got package 5043 on my screen. What would you like me to do with it? I can check its status or redirect it — if you want a redirect, tell me the new destination and the security code.'

In [124]:
import requests

check_package_url = f'{settings.hub_base_url}/api/packages'
response = requests.post(
    url=check_package_url,
    json={
        'apikey': settings.ai_devs_hub_api_key,
        'action': 'check',
        'packageid': 'PKG12345678',
        'destination': 'PWR6132PL',
        'code': 'dasf'
    }
)

response.json()

{'code': -216,
 'message': 'Invalid "packageid" format. Expected "PKG" followed by 8 digits.'}